# YOLO: segmentación con COCO8-Seg, TensorBoard y reanudación en Colab

Este notebook entrena **YOLO11n-seg** mediante ajuste fino, registra el progreso
en TensorBoard y conserva los checkpoints en Google Drive para continuar en otra
sesión de Colab. El código se puede editar y versionar como `YOLO.ipynb` en el repositorio.

| Criterio de la actividad | Evidencia que genera el notebook |
| --- | --- |
| Entrenamiento (25 puntos) | Pérdidas por época, `results.csv`, `best.pt` y `last.pt` |
| Métricas de entrenamiento y prueba (25 puntos) | Tabla de train/val/test para cajas y máscaras, curvas y predicciones |
| TensorBoard durante el entrenamiento (25 puntos) | Panel iniciado **antes** de entrenar y archivos de eventos persistentes |
| Detener y continuar en otra sesión de Colab (25 puntos) | `resume.pt` con optimizador, época, historial y registro de sesiones |

**Dataset elegido:** el enunciado dice COCO8-Seg, aunque el enlace de Kaggle
corresponde al COCO 2017 completo. Se utiliza el **COCO8-Seg oficial**, de ocho
imágenes (aproximadamente 1 MB). No se descarga el dataset grande de Kaggle.

COCO8-Seg originalmente contiene 4 imágenes de entrenamiento y 4 de validación;
no incluye test. Aquí se mantienen las 4 de entrenamiento y se separan las otras
4 en **2 de validación y 2 de prueba**, de forma determinista. Las tres particiones
no comparten archivos durante este ajuste fino.

Es un ejercicio de funcionamiento: una prueba de dos imágenes no permite estimar
el rendimiento general. Además, los pesos iniciales fueron preentrenados en COCO;
la prueba no es independiente de ese preentrenamiento.


## Cómo ejecutarlo y entregar el enlace

1. En [Google Colab](https://colab.research.google.com/), usa **Archivo > Subir notebook**
   para abrir este archivo y guarda una copia en tu Drive.
2. Selecciona **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4**
   si está disponible. También funciona con CPU, con mayor tiempo de ejecución.
3. Ejecuta las celdas en orden y autoriza el montaje de **tu** Google Drive.
4. TensorBoard se abre antes de entrenar. Al completar la época 5, el código pausa
   intencionalmente y muestra la ubicación del checkpoint.
5. Para demostrar una segunda sesión, usa **Entorno de ejecución > Desconectar y
   eliminar entorno de ejecución**. Vuelve a conectar y ejecuta el notebook desde el
   principio con el **mismo `NOMBRE_EXPERIMENTO`**. El modo `auto` carga el checkpoint
   de Drive y continúa desde la época 6, conservando el optimizador y el historial.
6. Al finalizar, ejecuta las métricas, las predicciones y el resumen. Usa **Compartir**
   en Colab para obtener el enlace de lectura que solicita la actividad. Pégalo en
   `ENLACE_COLAB` en la configuración y vuelve a ejecutar la celda del resumen.
7. Descarga el `.ipynb` con sus salidas y reemplaza la copia del repositorio.

**Edición local:** puedes seguir modificando este archivo en VS Code. Para ejecutar
las celdas desde VS Code necesitas seleccionar un kernel remoto adecuado. El kernel
de Kaggle sirve para probar el código, pero la demostración de persistencia y el
enlace que pide la rúbrica deben hacerse en **Colab**.

En Kaggle los resultados se guardan en `/kaggle/working`; no se promete que esa ruta
sobreviva a otra sesión. En Colab se guardan directamente en Google Drive.
No ejecutes dos entrenamientos simultáneos con el mismo nombre de experimento.


## 1. Comprobar el entorno e instalar dependencias

Esta comprobación se ejecuta antes de instalar o descargar archivos. Protege la
computadora local de descargas involuntarias. Se fija la versión de Ultralytics
para que el checkpoint use la misma implementación en ambas sesiones.


In [ ]:
from pathlib import Path
import os
import sys

EN_COLAB = Path("/content").is_dir()
EN_KAGGLE = Path("/kaggle/input").is_dir()

if not (EN_COLAB or EN_KAGGLE):
    raise RuntimeError(
        "Selecciona el kernel remoto o abre este notebook en Google Colab. "
        "Se detuvo antes de instalar paquetes o descargar datos localmente."
    )

print("Entorno:", "Google Colab" if EN_COLAB else "Kaggle (prueba del código)")
print("Python:", sys.version.split()[0])


In [ ]:
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "ultralytics==8.3.221", "tensorboard==2.20.0", "numpy<2.3", "pandas>=2.2,<3",
])
print("Dependencias instaladas en el kernel actual.")


## 2. Almacenamiento persistente

El dataset pequeño se puede volver a descargar en cada sesión. Lo que debe
conservarse es el **estado del entrenamiento**: checkpoints, CSV y eventos de
TensorBoard. En Colab se escriben en `MyDrive/IA_avanzada_YOLO`.


In [ ]:
if EN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    RAIZ_TRABAJO = Path("/content")
    RAIZ_PERSISTENTE = Path("/content/drive/MyDrive/IA_avanzada_YOLO")
else:
    RAIZ_TRABAJO = Path("/kaggle/working")
    RAIZ_PERSISTENTE = RAIZ_TRABAJO / "IA_avanzada_YOLO"
    print("Prueba en Kaggle: usa Colab + Drive para demostrar otra sesión.")

RAIZ_PERSISTENTE.mkdir(parents=True, exist_ok=True)
os.chdir(RAIZ_TRABAJO)
print("Resultados persistentes:", RAIZ_PERSISTENTE)


## 3. Configuración del experimento

`MODO = "auto"` inicia si no existe checkpoint y reanuda si encuentra uno. Para
comenzar otro experimento, cambia `NOMBRE_EXPERIMENTO`; no borres el anterior.
`EPOCAS_TOTAL` es el objetivo total de ambas sesiones, no las épocas adicionales.
La pausa automática solo se aplica al entrenamiento nuevo.

Se usa una tasa de aprendizaje pequeña y se congelan las primeras diez capas del
backbone para hacer un ajuste fino conservador sobre un conjunto de solo cuatro
imágenes de entrenamiento. El resto de los parámetros entrenables sí se actualiza.


In [ ]:
import hashlib
import json
import platform
import socket
import uuid
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import display, Markdown
from ultralytics import YOLO, settings

NOMBRE_EXPERIMENTO = "yolo11n_coco8_seg"
MODO = "auto"                       # auto, nuevo o reanudar
EPOCAS_TOTAL = 30
PAUSAR_TRAS_EPOCA = 5                # None permite entrenar sin pausa automática
TAMANO_IMAGEN = 320
LOTE = 2
SEMILLA = 42
MODELO_INICIAL = "yolo11n-seg.pt"
TASA_APRENDIZAJE = 0.0001
CAPAS_CONGELADAS = 10                 # Conservar parte del backbone preentrenado
ENLACE_COLAB = ""                    # Pega aquí el enlace de lectura al entregar

assert MODO in {"auto", "nuevo", "reanudar"}
assert EPOCAS_TOTAL > 1
assert PAUSAR_TRAS_EPOCA is None or 0 < PAUSAR_TRAS_EPOCA < EPOCAS_TOTAL
assert NOMBRE_EXPERIMENTO and Path(NOMBRE_EXPERIMENTO).name == NOMBRE_EXPERIMENTO

DISPOSITIVO = 0 if torch.cuda.is_available() else "cpu"
DIRECTORIO_RUNS = RAIZ_PERSISTENTE / "runs"
DIRECTORIO_RUN = DIRECTORIO_RUNS / NOMBRE_EXPERIMENTO
DIRECTORIO_RUN.mkdir(parents=True, exist_ok=True)
CHECKPOINT = DIRECTORIO_RUN / "weights" / "resume.pt"
ESTADO_JSON = DIRECTORIO_RUN / "estado.json"
HISTORIAL_SESIONES = DIRECTORIO_RUN / "sesiones.jsonl"
CONFIG_DATOS = RAIZ_TRABAJO / "yolo_datos" / NOMBRE_EXPERIMENTO
CONFIG_DATOS.mkdir(parents=True, exist_ok=True)
DATA_YAML = CONFIG_DATOS / "coco8_seg_train_val_test.yaml"

ID_SESION = uuid.uuid4().hex[:12]
identidad_runtime = f"{socket.gethostname()}:{os.getpid()}"
ID_RUNTIME = hashlib.sha256(identidad_runtime.encode()).hexdigest()[:12]

# Activar el callback oficial antes de crear el entrenador.
settings.update({
    "tensorboard": True, "wandb": False, "mlflow": False, "comet": False,
    "datasets_dir": str(RAIZ_TRABAJO / "datasets"),
})

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Dispositivo:", torch.cuda.get_device_name(0) if DISPOSITIVO == 0 else "CPU")
print("Experimento:", DIRECTORIO_RUN)
print("Sesión:", ID_SESION, "| Runtime:", ID_RUNTIME)


## 4. COCO8-Seg y separación de prueba

Se descarga únicamente COCO8-Seg con sus polígonos de segmentación. Se crean
listas de archivos, sin duplicar las imágenes: train = 4, validación = 2, test = 2.
El test se usa al terminar; `best.pt` se selecciona con la validación, no con test.
El manifiesto persistente permite comprobar que ambas sesiones usan los mismos datos.


In [ ]:
from ultralytics.data.utils import check_det_dataset

datos_originales = check_det_dataset("coco8-seg.yaml")

def listar_imagenes(carpeta):
    extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sorted(p.resolve() for p in Path(carpeta).iterdir()
                  if p.suffix.lower() in extensiones)

imagenes_train = listar_imagenes(datos_originales["train"])
restantes = listar_imagenes(datos_originales["val"])
assert len(imagenes_train) == 4 and len(restantes) == 4, "Se esperaba COCO8-Seg."
imagenes_val = restantes[:2]
imagenes_test = restantes[2:]

grupos = {"train": imagenes_train, "val": imagenes_val, "test": imagenes_test}
conjuntos = [set(p.name for p in lista) for lista in grupos.values()]
assert not (conjuntos[0] & conjuntos[1] or conjuntos[0] & conjuntos[2]
            or conjuntos[1] & conjuntos[2]), "Hay imágenes repetidas entre particiones."

manifiesto = {}
configuracion_datos = {"path": str(datos_originales["path"]),
                       "names": datos_originales["names"]}
for split, lista in grupos.items():
    archivo_lista = CONFIG_DATOS / f"{split}.txt"
    archivo_lista.write_text("\n".join(p.as_posix() for p in lista) + "\n", encoding="utf-8")
    configuracion_datos[split] = str(archivo_lista)
    manifiesto[split] = []
    for imagen in lista:
        etiqueta = Path(str(imagen).replace(f"{os.sep}images{os.sep}",
                                           f"{os.sep}labels{os.sep}")).with_suffix(".txt")
        assert etiqueta.is_file(), f"Falta la etiqueta: {etiqueta}"
        for linea in etiqueta.read_text().splitlines():
            assert len(linea.split()) >= 7, f"Se esperaba un polígono: {etiqueta}"
        manifiesto[split].append({
            "archivo": imagen.name,
            "sha256_imagen": hashlib.sha256(imagen.read_bytes()).hexdigest(),
            "sha256_etiqueta": hashlib.sha256(etiqueta.read_bytes()).hexdigest(),
        })

DATA_YAML.write_text(yaml.safe_dump(configuracion_datos, sort_keys=False), encoding="utf-8")
ruta_manifiesto = DIRECTORIO_RUN / "particiones.json"
if ruta_manifiesto.exists():
    assert json.loads(ruta_manifiesto.read_text()) == manifiesto, "Las particiones cambiaron."
else:
    ruta_manifiesto.write_text(json.dumps(manifiesto, indent=2), encoding="utf-8")

display(pd.DataFrame([
    {"partición": split, "imágenes": len(lista), "archivos": ", ".join(p.name for p in lista)}
    for split, lista in grupos.items()
]))
print("Configuración de datos:", DATA_YAML)


## 5. Abrir TensorBoard ANTES del entrenamiento

El panel puede mostrar que aún no hay datos hasta terminar la primera época.
Déjalo abierto y ejecuta la siguiente sección: sus gráficos se actualizan mientras
entrena. Busca pérdidas `train/` y `val/`, tasas de aprendizaje y métricas de cajas
`(B)` y máscaras `(M)`. Los eventos permanecen en la carpeta de Drive del experimento.

Si el editor remoto de VS Code no muestra el panel, abre este mismo notebook en
Colab y ejecuta esta celda allí para la evidencia que pide la rúbrica.


In [ ]:
ipython = get_ipython()
ipython.run_line_magic("load_ext", "tensorboard")
ipython.run_line_magic(
    "tensorboard", f'--logdir "{DIRECTORIO_RUN}" --reload_interval 2'
)


## 6. Checkpoints y pausa recuperable

`last.pt` y `best.pt` son los archivos habituales de Ultralytics. Al finalizar un
entrenamiento, la librería puede quitarles el optimizador. Por eso, **después de cada
guardado** se conserva además `resume.pt`, una copia del checkpoint completo con
optimizador y número de época. Se escribe primero un temporal y luego se reemplaza
el archivo, para conservar la copia anterior si una escritura se interrumpe.

La pausa de demostración ocurre **al comenzar la siguiente época**, cuando la
anterior ya terminó de guardar sus métricas y checkpoint. No se usa `trainer.stop`,
que finalizaría normalmente el entrenamiento. Una desconexión imprevista recupera
la última época completa guardada; el trabajo a mitad de una época se pierde.


In [ ]:
import shutil

def guardar_json_atomico(ruta, datos):
    temporal = ruta.with_suffix(ruta.suffix + ".tmp")
    temporal.write_text(json.dumps(datos, indent=2, ensure_ascii=False), encoding="utf-8")
    temporal.replace(ruta)

def registrar_sesion(evento, **datos):
    registro = {
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
        "sesion": ID_SESION, "runtime": ID_RUNTIME,
        "entorno": "colab" if EN_COLAB else "kaggle",
        "evento": evento, **datos,
    }
    with HISTORIAL_SESIONES.open("a", encoding="utf-8") as archivo:
        archivo.write(json.dumps(registro, ensure_ascii=False) + "\n")

def guardar_checkpoint_completo(trainer):
    temporal = CHECKPOINT.with_suffix(".pt.tmp")
    shutil.copy2(trainer.last, temporal)
    temporal.replace(CHECKPOINT)
    guardar_json_atomico(ESTADO_JSON, {
        "epocas_completadas": trainer.epoch + 1,
        "epocas_objetivo": trainer.epochs,
        "checkpoint": str(CHECKPOINT),
        "sesion": ID_SESION, "runtime": ID_RUNTIME,
        "ultralytics": ultralytics.__version__, "torch": str(torch.__version__),
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
    })

class PausaParaOtraSesion(Exception):
    pass

def pausar_despues_de_checkpoint(trainer):
    if (PAUSAR_TRAS_EPOCA is not None
            and trainer.epoch >= PAUSAR_TRAS_EPOCA
            and CHECKPOINT.is_file()):
        raise PausaParaOtraSesion(
            f"Pausa después de {PAUSAR_TRAS_EPOCA} épocas completas. "
            "Conecta una nueva sesión de Colab y ejecuta desde el principio."
        )

configuracion_experimento = {
    "modelo": MODELO_INICIAL, "epocas": EPOCAS_TOTAL, "imgsz": TAMANO_IMAGEN,
    "batch": LOTE, "seed": SEMILLA, "ultralytics": ultralytics.__version__,
    "dataset": "coco8-seg", "particiones": {k: len(v) for k, v in grupos.items()},
    "lr0": TASA_APRENDIZAJE, "freeze": CAPAS_CONGELADAS,
}
config_guardada = DIRECTORIO_RUN / "configuracion.json"
if config_guardada.exists():
    assert json.loads(config_guardada.read_text()) == configuracion_experimento, (
        "La configuración cambió. Usa los valores anteriores o un nuevo NOMBRE_EXPERIMENTO."
    )
else:
    guardar_json_atomico(config_guardada, configuracion_experimento)

# Registrar las versiones reales ayuda a reproducir la segunda sesión.
versiones = {
    "python": platform.python_version(), "torch": str(torch.__version__),
    "ultralytics": ultralytics.__version__,
}
guardar_json_atomico(DIRECTORIO_RUN / f"entorno_{ID_SESION}.json", versiones)


## 7. Entrenar o reanudar automáticamente

El objetivo es completar 30 épocas entre las sesiones. La primera ejecución se
pausa tras 5; la segunda completa las restantes. Para una interrupción manual,
espera al menos una época completa y usa el botón de detener. Antes de eliminar
el runtime, confirma que existen `resume.pt` y `estado.json` en Drive.

Si solo vuelves a ejecutar la celda, también reanuda; para demostrar **otra sesión**
debes desconectar/eliminar el runtime de Colab y volver a conectar. El registro
compara la identidad del proceso remoto con la que guardó el checkpoint.


In [ ]:
REANUDAR = MODO == "reanudar" or (MODO == "auto" and CHECKPOINT.is_file())
if MODO == "nuevo" and CHECKPOINT.exists():
    raise FileExistsError("Ya existe este experimento. Cambia NOMBRE_EXPERIMENTO.")
if REANUDAR and not CHECKPOINT.is_file():
    raise FileNotFoundError(f"No hay checkpoint para reanudar: {CHECKPOINT}")

epocas_previas = 0
runtime_anterior = None
if REANUDAR:
    # Archivo creado por este notebook; no cargar checkpoints de origen desconocido.
    checkpoint_guardado = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    epocas_previas = int(checkpoint_guardado["epoch"]) + 1
    if checkpoint_guardado.get("optimizer") is None:
        raise RuntimeError("El checkpoint no conserva el optimizador y no permite reanudación real.")
    if ESTADO_JSON.exists():
        runtime_anterior = json.loads(ESTADO_JSON.read_text())["runtime"]
    del checkpoint_guardado

ENTRENAMIENTO_COMPLETO = epocas_previas >= EPOCAS_TOTAL
if ENTRENAMIENTO_COMPLETO:
    print(f"El entrenamiento ya completó {epocas_previas} épocas. Continúa con las métricas.")
else:
    registrar_sesion(
        "reanudar" if REANUDAR else "iniciar",
        desde_epoca=epocas_previas, objetivo=EPOCAS_TOTAL,
        runtime_anterior=runtime_anterior,
        otro_runtime=bool(runtime_anterior and runtime_anterior != ID_RUNTIME),
    )
    print("Reanudando" if REANUDAR else "Iniciando", "desde la época", epocas_previas + 1)
    modelo = YOLO(str(CHECKPOINT) if REANUDAR else MODELO_INICIAL)
    modelo.add_callback("on_model_save", guardar_checkpoint_completo)
    if not REANUDAR:
        modelo.add_callback("on_train_epoch_start", pausar_despues_de_checkpoint)

    try:
        if REANUDAR:
            resultado_entrenamiento = modelo.train(
                resume=True, data=str(DATA_YAML), device=DISPOSITIVO, workers=0,
            )
        else:
            resultado_entrenamiento = modelo.train(
                data=str(DATA_YAML), epochs=EPOCAS_TOTAL, imgsz=TAMANO_IMAGEN,
                batch=LOTE, device=DISPOSITIVO, workers=0,
                project=str(DIRECTORIO_RUNS), name=NOMBRE_EXPERIMENTO, exist_ok=True,
                optimizer="AdamW", lr0=TASA_APRENDIZAJE, freeze=CAPAS_CONGELADAS,
                warmup_bias_lr=TASA_APRENDIZAJE, seed=SEMILLA, deterministic=True,
                save=True, save_period=5, val=True, plots=True, patience=0,
                amp=DISPOSITIVO != "cpu", close_mosaic=0, verbose=True,
            )
        registrar_sesion("finalizado", epocas_completadas=EPOCAS_TOTAL)
    except (PausaParaOtraSesion, KeyboardInterrupt) as interrupcion:
        print(str(interrupcion) or "Interrupción manual: se conserva la última época completa.")
        registrar_sesion("pausado", checkpoint_disponible=CHECKPOINT.is_file())
    finally:
        # Una pausa por excepción no ejecuta el cierre normal del logger de la librería.
        from ultralytics.utils.callbacks import tensorboard as callback_tb

        writer = getattr(callback_tb, "WRITER", None)
        if writer is not None:
            writer.flush()
            writer.close()
            callback_tb.WRITER = None

    if CHECKPOINT.exists():
        estado = json.loads(ESTADO_JSON.read_text())
        ENTRENAMIENTO_COMPLETO = estado["epocas_completadas"] >= EPOCAS_TOTAL
        display(pd.DataFrame([estado]))
        print("Checkpoint recuperable:", CHECKPOINT)
    else:
        print("No hay checkpoint todavía: debe completarse al menos una época.")


## 8. Progreso de entrenamiento y eventos de TensorBoard

Estas celdas también se pueden ejecutar después de la pausa para mostrar lo
guardado. `train/*loss` son pérdidas de entrenamiento; las métricas de validación
por época están en las columnas `metrics/`. La evaluación final sobre train y test
se realiza por separado en la siguiente sección.


In [ ]:
ruta_csv = DIRECTORIO_RUN / "results.csv"
if ruta_csv.exists():
    historial = pd.read_csv(ruta_csv)
    historial.columns = historial.columns.str.strip()
    display(historial)
    columnas_perdida = [c for c in historial if c.startswith(("train/", "val/")) and "loss" in c]
    columnas_metricas = [c for c in historial if c.startswith("metrics/")]
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    historial.plot(x="epoch", y=columnas_perdida, ax=axes[0], title="Pérdidas por época")
    historial.plot(x="epoch", y=columnas_metricas, ax=axes[1], title="Métricas de validación")
    for ax in axes:
        ax.set_xlabel("Época")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("Épocas registradas:", historial["epoch"].tolist())
else:
    print("Ejecuta al menos una época completa para generar results.csv.")

eventos = sorted(DIRECTORIO_RUN.glob("events.out.tfevents.*"))
print("Archivos de TensorBoard:", len(eventos))
for archivo in eventos:
    print(archivo.name, "| bytes:", archivo.stat().st_size)


## 9. Métricas finales de entrenamiento, validación y prueba

Se carga `best.pt`, elegido por la validación, y se evalúan las tres particiones.
Se reportan precisión, recall, mAP@0.50 y mAP@0.50:0.95 para **cajas y máscaras**.
El test no decide qué checkpoint es mejor. Las métricas se calculan realmente;
no se introducen resultados manuales. Esta sección se omite hasta completar el
objetivo de épocas.


In [ ]:
tabla_metricas = None
modelo_evaluado = None
if not ENTRENAMIENTO_COMPLETO:
    print("Evaluación final pendiente: reanuda en la segunda sesión y completa las épocas.")
else:
    mejor_peso = DIRECTORIO_RUN / "weights" / "best.pt"
    if not mejor_peso.is_file():
        raise FileNotFoundError(f"No se encontró el mejor modelo: {mejor_peso}")
    modelo_evaluado = YOLO(str(mejor_peso))
    filas = []
    resultados_metricas = {}
    for split in ("train", "val", "test"):
        metricas = modelo_evaluado.val(
            data=str(DATA_YAML), split=split, imgsz=TAMANO_IMAGEN,
            batch=LOTE, device=DISPOSITIVO, workers=0, plots=True,
            project=str(DIRECTORIO_RUN / "evaluacion"), name=split, exist_ok=True,
        )
        resultados_metricas[split] = {k: float(v) for k, v in metricas.results_dict.items()}
        for tipo, valores in (("cajas", metricas.box), ("máscaras", metricas.seg)):
            filas.append({
                "partición": split, "tipo": tipo, "imágenes": len(grupos[split]),
                "precisión": float(valores.mp), "recall": float(valores.mr),
                "mAP50": float(valores.map50), "mAP50-95": float(valores.map),
            })
    tabla_metricas = pd.DataFrame(filas)
    display(tabla_metricas.round(4))
    tabla_metricas.to_csv(DIRECTORIO_RUN / "metricas_finales.csv", index=False)
    guardar_json_atomico(DIRECTORIO_RUN / "metricas_finales.json", resultados_metricas)
    print("Métricas y gráficas guardadas en:", DIRECTORIO_RUN)


## 10. Predicciones sobre las imágenes de prueba

Las máscaras y cajas muestran qué predijo el modelo; no sustituyen las métricas.
Se guardan las imágenes anotadas para documentar la entrega.


In [ ]:
if modelo_evaluado is None:
    print("Completa primero el entrenamiento y la evaluación final.")
else:
    from PIL import Image

    carpeta_predicciones = DIRECTORIO_RUN / "predicciones_test"
    carpeta_predicciones.mkdir(exist_ok=True)
    predicciones = modelo_evaluado.predict(
        source=[str(p) for p in imagenes_test], imgsz=TAMANO_IMAGEN,
        conf=0.25, device=DISPOSITIVO, save=False, verbose=False,
    )
    fig, axes = plt.subplots(1, len(predicciones), figsize=(14, 6), squeeze=False)
    for ax, prediccion in zip(axes[0], predicciones):
        imagen_rgb = prediccion.plot()[..., ::-1]
        ax.imshow(imagen_rgb)
        ax.set_title(Path(prediccion.path).name)
        ax.axis("off")
        Image.fromarray(imagen_rgb).save(carpeta_predicciones / Path(prediccion.path).name)
    plt.tight_layout()
    plt.show()


## 11. Comprobar la reanudación y generar el resumen de entrega

Una fila `reanudar` con `desde_epoca > 0` muestra que se recuperó un checkpoint.
`otro_runtime = true` y `entorno = colab` documentan la continuación desde otro
proceso remoto de Colab. Si solo se usó Kaggle o el mismo runtime, el resumen lo
marca pendiente. Guarda también una captura del panel de TensorBoard durante el
entrenamiento y la salida que anuncia la época desde la que reanuda.


In [ ]:
registros = []
if HISTORIAL_SESIONES.exists():
    registros = [json.loads(linea) for linea in HISTORIAL_SESIONES.read_text(encoding="utf-8").splitlines()
                 if linea.strip()]
    display(pd.DataFrame(registros))

reanudaciones_colab = [r for r in registros if r["evento"] == "reanudar"
                       and r.get("desde_epoca", 0) > 0
                       and r.get("otro_runtime") and r["entorno"] == "colab"]

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

pasos_tensorboard = set()
tags_tensorboard = set()
for archivo in DIRECTORIO_RUN.glob("events.out.tfevents.*"):
    acumulador = EventAccumulator(str(archivo), size_guidance={"scalars": 0})
    acumulador.Reload()
    for tag in acumulador.Tags().get("scalars", []):
        tags_tensorboard.add(tag)
        pasos_tensorboard.update(evento.step for evento in acumulador.Scalars(tag))
print("Tags de TensorBoard:", sorted(tags_tensorboard))
print("Épocas con eventos:", sorted(pasos_tensorboard))

comprobaciones = {
    "entrenamiento_completo": ENTRENAMIENTO_COMPLETO,
    "metricas_train_val_test": tabla_metricas is not None,
    "tensorboard_con_metricas": bool(tags_tensorboard),
    "reanudo_en_otro_runtime_colab": bool(reanudaciones_colab),
    "enlace_colab": ENLACE_COLAB,
}
guardar_json_atomico(DIRECTORIO_RUN / "comprobaciones_entrega.json", comprobaciones)

lineas_resumen = [
    "# YOLO: resumen de ejecución", "",
    f"Modelo: {MODELO_INICIAL}", "Dataset: COCO8-Seg (4 train, 2 val, 2 test)",
    f"Objetivo: {EPOCAS_TOTAL} épocas", "",
]
for criterio, valor in comprobaciones.items():
    lineas_resumen.append(f"- {criterio}: {valor if valor else 'PENDIENTE'}")
lineas_resumen.extend(["", "## Métricas finales", ""])
if tabla_metricas is not None:
    lineas_resumen.append("```text\n" + tabla_metricas.round(4).to_string(index=False) + "\n```")
else:
    lineas_resumen.append("Pendientes de completar el entrenamiento.")
lineas_resumen.extend([
    "", "## Alcance",
    "Ejercicio con ocho imágenes. El test de dos imágenes no demuestra generalización.",
    "Los pesos iniciales fueron preentrenados en COCO.",
    "La existencia de eventos no sustituye la captura del panel durante el entrenamiento.",
])
resumen = "\n".join(lineas_resumen)
(DIRECTORIO_RUN / "resumen_ejecucion.md").write_text(resumen, encoding="utf-8")
display(Markdown(resumen))


## Entrega

La entrega requiere el **enlace de Colab** donde se pueda abrir y ejecutar el
código. No basta con el enlace del repositorio o del notebook de Kaggle.

- Comparte el notebook de Colab con permiso de lectura para quien lo evalúe.
- Conserva las salidas del entrenamiento, de la segunda sesión y de las métricas.
- Incluye la captura de TensorBoard mientras entrenaba.
- Ejecuta la sección de comprobaciones: los cuatro criterios deben estar completos.
- Descarga la copia final para conservarla en este repositorio.

Los checkpoints y datos permanecen en Drive o en el runtime; no necesitas subir
pesos `.pt`, datasets ni eventos de TensorBoard a Git.

### Referencias

- [Entrenamiento y reanudación de Ultralytics](https://docs.ultralytics.com/modes/train/)
- [COCO8-Seg: estructura y descarga](https://docs.ultralytics.com/datasets/segment/coco8-seg/)
- [Integración con TensorBoard](https://docs.ultralytics.com/integrations/tensorboard/)
- [TensorBoard dentro de notebooks](https://www.tensorflow.org/tensorboard/tensorboard_in_notebooks)
- [Versión fijada de Ultralytics](https://pypi.org/project/ultralytics/8.3.221/)
